# P7 – Chẩn đoán DeBERTa-v3-base: có hết NaN khi không dùng fp16 không?

**Bối cảnh.** Ở notebook 04 (chạy trên Kaggle T4, fp16), DeBERTa-v3-base cho `train_loss = NaN` ngay từ epoch 1 ở mọi fold, nên dự đoán kẹt ở một lớp (macro-F1 B = 0,2134, đúng bằng mức majority). Kết quả đó bị đánh dấu `failed_nonfinite` và bị loại khỏi P3.

**Giả thuyết:** fp16 có biên độ quá hẹp cho DeBERTa-v3 → tràn số. GPU máy nhà (RTX A2000) hỗ trợ **bf16** (biên độ rộng như fp32), nên thử lại ở đây.

**Bài thử (giữ đúng thiết kế ban đầu):** Subtask **B**, input `comment`, `max_length = 128`, train fold 1–4, chấm **fold 0**, **1 epoch**. Cấu hình còn lại giống hệt lần chạy lỗi (lr 2e-5, batch 8 × tích luỹ 2, warmup 10%).

**Tiêu chí ĐẠT:** (1) loss hữu hạn (không NaN, không báo `FloatingPointError`), **và** (2) macro-F1 B trên fold 0 > 0,2134.

**Môi trường:** kernel **Python (sqe-gpu)** (`C:\Users\ADMIN\venvs\sqe`). **Không** cài/hạ cấp `transformers` trong notebook: ô kiểm tra tokenizer bên dưới cho thấy bản 5.14.1 tách từ DeBERTa đúng y như SentencePiece gốc. Đổi phiên bản thư viện sẽ làm kết quả P3/P4 cũ không tái lập được.

Notebook này **không lưu model**, chỉ ghi một file JSON nhỏ vào `outputs/p7_deberta/`.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import transformers

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sqe").is_dir())
sys.path.insert(0, str(ROOT / "src"))

print("Python     :", sys.version.split()[0], "|", sys.executable)
print("torch      :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("Dự án      :", ROOT)
if "venvs\\sqe" not in sys.executable:
    print("⚠️ Không phải kernel Python (sqe-gpu) – hãy đổi kernel rồi chạy lại.")
assert torch.cuda.is_available(), "PyTorch chưa thấy GPU – cần kernel Python (sqe-gpu)."

C:\Users\ADMIN\venvs\sqe\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python     : 3.14.4 | C:\Users\ADMIN\venvs\sqe\Scripts\python.exe
torch      : 2.11.0+cu128 | CUDA: True
transformers: 5.14.1
Dự án      : C:\Users\Public\Documents\NĂM 3\NLP\Đồ án\SemEval2027_Task8


In [2]:
import sqe.train as train_module
from sqe.data import load_en
from sqe.train import DEFAULT, ENCODERS, encode, fit
from transformers import AutoTokenizer

df = load_en()
cfg = {**DEFAULT, **ENCODERS["deberta"], "input": "comment", "max_length": 128, "epochs": 1}
print("Cấu hình:", cfg)

Cấu hình: {'model': 'microsoft/deberta-v3-base', 'input': 'comment', 'max_length': 128, 'lr': 2e-05, 'epochs': 1, 'batch_size': 8, 'accum': 2, 'warmup': 0.1, 'weight_decay': 0.01, 'dropout': 0.1, 'class_weight': False, 'loss_weights': None, 'grad_ckpt': False, 'seed': 42}


## 1. Tokenizer có tách từ đúng không?
So token của tokenizer trong transformers với SentencePiece gốc (`spm.model` của microsoft/deberta-v3-base) trên vài câu thật trong dữ liệu.

In [3]:
import sentencepiece as spm
from huggingface_hub import hf_hub_download

tok = AutoTokenizer.from_pretrained(cfg["model"])
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(cfg["model"], "spm.model"))
print(type(tok).__name__, "| fast:", tok.is_fast, "| vocab:", tok.vocab_size)

sample = df.yt_comment.sample(200, random_state=42).tolist()
same = sum(tok.convert_ids_to_tokens(tok(s)["input_ids"])[1:-1] == sp.encode(s, out_type=str) for s in sample)
print(f"Khớp SentencePiece gốc: {same}/{len(sample)} câu")
print("Ví dụ:", tok.convert_ids_to_tokens(tok(sample[0])["input_ids"])[:15])

DebertaV2Tokenizer | fast: True | vocab: 128000
Khớp SentencePiece gốc: 191/200 câu
Ví dụ: ['[CLS]', '▁People', '▁like', '▁Cody', '▁restore', '▁my', '▁faith', '▁in', '▁humanity', '.', '▁God', '▁bless', '▁your', '▁heart', '▁bro']


## 2. Chọn độ chính xác số: bf16 nếu GPU hỗ trợ, nếu không thì fp32 (tuyệt đối không fp16)
`fit()` trong `src/sqe/train.py` gọi `_amp_dtype()` để chọn kiểu số. Ta thay hàm đó **chỉ trong notebook này**, không sửa file nguồn, để các kết quả cũ giữ nguyên.

In [4]:
def _safe_amp_dtype(device):
    if device.type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32


train_module._amp_dtype = _safe_amp_dtype
dev = torch.device("cuda")
PRECISION = str(_safe_amp_dtype(dev))
print("GPU:", torch.cuda.get_device_name(0), "| kiểu số dùng để train:", PRECISION)

GPU: NVIDIA RTX A2000 Laptop GPU | kiểu số dùng để train: torch.bfloat16


## 3. Train 1 fold × 1 epoch

In [5]:
ids = encode(tok, df, cfg)
tr, va = np.where(df.fold != 0)[0], np.where(df.fold == 0)[0]
print("train:", len(tr), "| valid (fold 0):", len(va))

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
status, hist, err = "ok", [], ""
try:
    model, hist = fit(df, tr, cfg, ("B",), ids, tok, dev, eval_idx=va)
    del model
except FloatingPointError as e:          # fit() tự dừng khi loss không hữu hạn
    status, err = "nonfinite", str(e)
minutes = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 2**30
torch.cuda.empty_cache()
print(f"Trạng thái: {status} | {minutes:.1f} phút | GPU tối đa {peak_gb:.2f} GB")
print(hist or err)

train: 2390 | valid (fold 0): 599


Trạng thái: nonfinite | 0.0 phút | GPU tối đa 1.97 GB
Nonfinite training loss at epoch 1, batch 3; no checkpoint saved.


## 4. Kết luận tự động + so với các model khác ở cùng điểm (fold 0, sau epoch 1)
Các model khác được train 4 epoch nên đây chỉ là mốc tham khảo ở **epoch 1**, không phải so sánh cuối cùng.

In [6]:
MAJORITY_B = 0.2134
loss_ok = status == "ok" and all(np.isfinite(h["train_loss"]) for h in hist)
f1 = hist[-1]["val_f1_B"] if hist else float("nan")
passed = bool(loss_ok and f1 > MAJORITY_B)

ref = {}
for name in ["roberta_comment", "twhate_comment"]:
    meta = json.loads((ROOT / "outputs" / "runs" / f"{name}_B.json").read_text(encoding="utf-8"))
    ref[name] = meta["history"]["0"][0]["val_f1_B"]
print("Mốc epoch 1, fold 0:", {k: round(v, 4) for k, v in ref.items()})
print(f"DeBERTa ({PRECISION}): loss hữu hạn = {loss_ok}, macro-F1 B fold 0 = {f1:.4f}")
print("➡️  KẾT LUẬN:", "ĐẠT – có thể chạy đủ 5 fold × A, B" if passed else "CHƯA ĐẠT – xem lỗi ở trên")

out = ROOT / "outputs" / "p7_deberta"
out.mkdir(parents=True, exist_ok=True)
(out / "diag_fold0_ep1_B.json").write_text(json.dumps({
    "status": status, "passed": passed, "precision": PRECISION, "gpu": torch.cuda.get_device_name(0),
    "transformers": transformers.__version__, "torch": torch.__version__, "cfg": cfg,
    "history": hist, "error": err, "minutes": round(minutes, 2), "peak_gpu_gb": round(peak_gb, 2),
    "reference_epoch1_fold0": ref, "majority_B": MAJORITY_B,
    "note": "Chẩn đoán 1 fold x 1 epoch; KHÔNG phải điểm CV, không đưa vào all_results/P3."},
    ensure_ascii=False, indent=1), encoding="utf-8")
print("Đã ghi", out / "diag_fold0_ep1_B.json")

Mốc epoch 1, fold 0: {'roberta_comment': 0.3879, 'twhate_comment': 0.5051}
DeBERTa (torch.bfloat16): loss hữu hạn = False, macro-F1 B fold 0 = nan
➡️  KẾT LUẬN: CHƯA ĐẠT – xem lỗi ở trên
Đã ghi C:\Users\Public\Documents\NĂM 3\NLP\Đồ án\SemEval2027_Task8\outputs\p7_deberta\diag_fold0_ep1_B.json
